# 02 - Generate `silver_obt.sql` with Jinja  (Ansh's homework, same idea as his `silver_obt` notebook)
The pipeline can't use Jinja directly, it needs a finished `.sql` file. This notebook:
1. holds the table/join config (`JINJA_CONFIG`)
2. renders the Jinja template (`for` loops, `if`, variables) into one `CREATE OR REFRESH STREAMING TABLE` query
3. writes the result straight into the pipeline's `transformations/silver_obt.sql`, no copy-paste

Run it **before** the pipeline (as a Job task). To add a lookup table, edit `JINJA_CONFIG` and re-run.
If `jinja2` is missing, run `%pip install jinja2` in a cell above first.

In [0]:
TARGET = "uber.silver.silver_obt"
PROJECT_FOLDER = "Uber_Project"          # your workspace folder that holds the pipeline
PIPELINE_FOLDER = "uber_rides_ingest"    # the pipeline's folder name

user = spark.sql("SELECT current_user()").first()[0]
OUT_FILE = f"/Workspace/Users/{user}/{PROJECT_FOLDER}/{PIPELINE_FOLDER}/transformations/silver_obt.sql"
print("will write to:", OUT_FILE)

will write to: /Workspace/Users/kaustavking2@gmail.com/Uber_Project/uber_rides_ingest/transformations/silver_obt.sql


In [0]:
# One entry per table. The FIRST entry is the streaming fact side, the rest are LEFT JOINed lookups.
JINJA_CONFIG = [
    {
        "table": "STREAM(uber.silver.stg_rides) WATERMARK booking_timestamp DELAY OF INTERVAL 3 MINUTES stg_rides",
        "select": ["stg_rides.*"],
        "where": "",
    },
    {
        "table": "uber.bronze.map_vehicle_makes map_vehicle_makes",
        "select": ["map_vehicle_makes.vehicle_make"],
        "on": "stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id",
        "where": "",
    },
    {
        "table": "uber.bronze.map_vehicle_types map_vehicle_types",
        "select": [
            "map_vehicle_types.vehicle_type",
            "map_vehicle_types.description",
            "map_vehicle_types.base_rate",
            "map_vehicle_types.per_mile",
            "map_vehicle_types.per_minute",
        ],
        "on": "stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id",
        "where": "",
    },
    {
        "table": "uber.bronze.map_ride_statuses map_ride_statuses",
        "select": ["map_ride_statuses.ride_status"],
        "on": "stg_rides.ride_status_id = map_ride_statuses.ride_status_id",
        "where": "",
    },
    {
        "table": "uber.bronze.map_payment_methods map_payment_methods",
        "select": [
            "map_payment_methods.payment_method",
            "map_payment_methods.is_card",
            "map_payment_methods.requires_auth",
        ],
        "on": "stg_rides.payment_method_id = map_payment_methods.payment_method_id",
        "where": "",
    },
    {
        "table": "uber.bronze.map_cities map_cities",
        "select": [
            "map_cities.city AS pickup_city",
            "map_cities.state",
            "map_cities.region",
            "map_cities.updated_at AS city_updated_at",
        ],
        "on": "stg_rides.pickup_city_id = map_cities.city_id",
        "where": "",
    },
    {
        "table": "uber.bronze.map_cancellation_reasons map_cancellation_reasons",
        "select": ["map_cancellation_reasons.cancellation_reason"],
        "on": "stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id",
        "where": "",
    },
]

In [0]:
TEMPLATE = """\
CREATE OR REFRESH STREAMING TABLE {{ target }}
AS
SELECT
{% for config in jinja_config %}
    {{ config.select | join(",\\n    ") }}{% if not loop.last %},{% endif %}

{% endfor %}
FROM
{% for config in jinja_config %}
{% if loop.first %}
    {{ config.table }}
{% else %}
LEFT JOIN {{ config.table }}
       ON {{ config.on }}
{% endif %}
{% endfor %}
{% if wheres %}
WHERE {{ wheres | join(" AND ") }}
{% endif %}
;
"""

In [0]:
from jinja2 import Environment

env = Environment(trim_blocks=True, lstrip_blocks=True)
wheres = [c["where"] for c in JINJA_CONFIG if c.get("where")]
rendered = env.from_string(TEMPLATE).render(target=TARGET, jinja_config=JINJA_CONFIG, wheres=wheres)
print(rendered)

CREATE OR REFRESH STREAMING TABLE uber.silver.silver_obt
AS
SELECT
    stg_rides.*,
    map_vehicle_makes.vehicle_make,
    map_vehicle_types.vehicle_type,
    map_vehicle_types.description,
    map_vehicle_types.base_rate,
    map_vehicle_types.per_mile,
    map_vehicle_types.per_minute,
    map_ride_statuses.ride_status,
    map_payment_methods.payment_method,
    map_payment_methods.is_card,
    map_payment_methods.requires_auth,
    map_cities.city AS pickup_city,
    map_cities.state,
    map_cities.region,
    map_cities.updated_at AS city_updated_at,
    map_cancellation_reasons.cancellation_reason
FROM
    STREAM(uber.silver.stg_rides) WATERMARK booking_timestamp DELAY OF INTERVAL 3 MINUTES stg_rides
LEFT JOIN uber.bronze.map_vehicle_makes map_vehicle_makes
       ON stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id
LEFT JOIN uber.bronze.map_vehicle_types map_vehicle_types
       ON stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id
LEFT JOIN uber.bronze.

In [0]:
# Write the rendered SQL into the pipeline folder (Python pathlib/os, as Ansh suggests)
from pathlib import Path

try:
    out = Path(OUT_FILE)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(rendered, encoding="utf-8")
    print("written:", OUT_FILE)
except Exception as e:
    print("Could not write the file automatically:", e)
    print("Fallback: copy the SQL printed above into the pipeline's silver_obt.sql by hand.")
    raise

written: /Workspace/Users/kaustavking2@gmail.com/Uber_Project/uber_rides_ingest/transformations/silver_obt.sql
